# Data Matching Investigation

This notebook investigates the data matching issue between `capstone2026v2.csv` and `all_skeleton_exports.csv`.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Data
Load the two CSV files into pandas DataFrames.

In [2]:
try:
    main_df = pd.read_csv('public/data/capstone2026v2.csv')
    export_df = pd.read_csv('public/data/all_skeleton_exports.csv')
    print("Files loaded successfully.")
    print(f"Main DF shape: {main_df.shape}")
    print(f"Export DF shape: {export_df.shape}")
except FileNotFoundError as e:
    print(e)

Files loaded successfully.
Main DF shape: (25868, 264)
Export DF shape: (10987, 25)


## 2. Inspect and Clean Data
Examine the data types and contents of the key columns that will be used for merging.

In [3]:
print("--- Main DataFrame Info ---")
main_df.info()
print("\n--- Export DataFrame Info ---")
export_df.info()

--- Main DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25868 entries, 0 to 25867
Columns: 264 entries, Name to hand
dtypes: bool(1), float64(258), object(5)
memory usage: 51.9+ MB

--- Export DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10987 entries, 0 to 10986
Data columns (total 25 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   PlayerId   10987 non-null  object 
 1   ShotId     10987 non-null  int64  
 2   Shot.Type  10987 non-null  object 
 3   hand       10987 non-null  object 
 4   is_make    10987 non-null  int64  
 5   feat_1     10987 non-null  object 
 6   orig_1     10987 non-null  float64
 7   rec_1      10987 non-null  float64
 8   delta_1    10987 non-null  float64
 9   feat_2     10987 non-null  object 
 10  orig_2     10987 non-null  float64
 11  rec_2      10987 non-null  float64
 12  delta_2    10987 non-null  float64
 13  feat_3     10987 non-null  object 
 14  orig_3     1

In [4]:
print("Unique values in main_df['Name'] (sample):", main_df['Name'].unique()[:5])
print("Unique values in export_df['PlayerId'] (sample):", export_df['PlayerId'].unique()[:5])

Unique values in main_df['Name'] (sample): ['Player 1' 'Player 2' 'Player 3' 'Player 4' 'Player 5']
Unique values in export_df['PlayerId'] (sample): ['Player 1' 'Player 10' 'Player 101' 'Player 102' 'Player 103']


The web application generates a shot index for each player. We need to replicate that logic here. We'll create a `shot_index` column which is a 0-based index that resets for each player.

In [5]:
main_df['shot_index'] = main_df.groupby('Name').cumcount()
# The ShotId in the export file is 1-based.
main_df['ShotId'] = (main_df['shot_index'] + 1).astype(str)

main_df[['Name', 'shot_index', 'ShotId']].head(10)

,Name,shot_index,ShotId
0,Player 1,0,1
1,Player 1,1,2
2,Player 1,2,3
3,Player 1,3,4
4,Player 1,4,5
5,Player 1,5,6
6,Player 1,6,7
7,Player 1,7,8
8,Player 1,8,9
9,Player 1,9,10


## 3. Attempt to Merge
Now, let's try to merge the two DataFrames based on the player identifier and the shot ID.

In [6]:
# Prepare for merge by ensuring data types and column names are consistent.
main_df_renamed = main_df.rename(columns={'Name': 'PlayerId'})

# Clean up whitespace and ensure types are strings for merging
main_df_renamed['PlayerId'] = main_df_renamed['PlayerId'].str.strip()
main_df_renamed['ShotId'] = main_df_renamed['ShotId'].astype(str).str.strip()

export_df['PlayerId'] = export_df['PlayerId'].astype(str).str.strip()
export_df['ShotId'] = export_df['ShotId'].astype(str).str.strip()

# Perform the merge
merged_df = pd.merge(
    main_df_renamed, 
    export_df, 
    on=['PlayerId', 'ShotId'], 
    how='inner'
)

print(f"Found {len(merged_df)} matching rows.")
if len(merged_df) > 0:
    print("\nSample of merged data:")
    display(merged_df[['PlayerId', 'ShotId', 'feat_1']].head())

Found 29 matching rows.

Sample of merged data:


,PlayerId,ShotId,feat_1
0,Player 1,3,BallVeloRelease__zwithin
1,Player 1,4,timeToMaxHipExtensionDominantPostHitch
2,Player 1,5,BallVeloRelease__zwithin
3,Player 1,8,BallVeloRelease__zwithin
4,Player 1,10,timeToMaxShoulderExtensionDomDiffHitch__zwithin


## 4. Merge All Individual Skeleton Export Files

Now we'll load all individual skeleton export files from the directory and combine them into a single, clean CSV.

In [7]:
import glob
import os

# Find all skeleton export files
skeleton_dir = 'public/data/skeleton_exports'
skeleton_files = sorted(glob.glob(os.path.join(skeleton_dir, 'skeleton_export_*.csv')))

print(f"Found {len(skeleton_files)} skeleton export files.")
print(f"First few files: {skeleton_files[:3]}")
print(f"Last few files: {skeleton_files[-3:]}")


Found 165 skeleton export files.
First few files: ['public/data/skeleton_exports/skeleton_export_Player 1.csv', 'public/data/skeleton_exports/skeleton_export_Player 10.csv', 'public/data/skeleton_exports/skeleton_export_Player 100.csv']
Last few files: ['public/data/skeleton_exports/skeleton_export_Player 97.csv', 'public/data/skeleton_exports/skeleton_export_Player 98.csv', 'public/data/skeleton_exports/skeleton_export_Player 99.csv']


In [11]:
# Load and combine all skeleton export files
all_exports = []

for file_path in skeleton_files:
    try:
        df = pd.read_csv(file_path)
        if len(df) > 0:  # Only add non-empty dataframes
            all_exports.append(df)
            print(f"Loaded {file_path}: {len(df)} rows")
        else:
            print(f"Skipped {file_path}: empty file")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

# Combine all dataframes
combined_exports_df = pd.concat(all_exports, ignore_index=True)
print(f"\nTotal rows after combining: {len(combined_exports_df)}")
print(f"Columns: {list(combined_exports_df.columns)}")
print(f"\nFirst few rows:")
print(combined_exports_df.head())

Loaded public/data/skeleton_exports/skeleton_export_Player 1.csv: 29 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 10.csv: 49 rows
Error loading public/data/skeleton_exports/skeleton_export_Player 100.csv: No columns to parse from file
Loaded public/data/skeleton_exports/skeleton_export_Player 101.csv: 40 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 102.csv: 697 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 103.csv: 137 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 104.csv: 59 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 105.csv: 28 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 106.csv: 27 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 107.csv: 15 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 108.csv: 27 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 109.csv: 24 rows
Loaded public/data/skeleton_exports/skeleton_export_Player 1

In [12]:
# Now test the merge with the main data using the combined exports
main_df_test = main_df.copy()
main_df_test['shot_index'] = main_df_test.groupby('Name').cumcount()
main_df_test['ShotId'] = (main_df_test['shot_index'] + 1).astype(str)

# Rename for merge
main_df_test = main_df_test.rename(columns={'Name': 'PlayerId'})
main_df_test['PlayerId'] = main_df_test['PlayerId'].str.strip()
main_df_test['ShotId'] = main_df_test['ShotId'].astype(str).str.strip()

combined_exports_df['PlayerId'] = combined_exports_df['PlayerId'].astype(str).str.strip()
combined_exports_df['ShotId'] = combined_exports_df['ShotId'].astype(str).str.strip()

# Test merge
test_merged = pd.merge(
    main_df_test,
    combined_exports_df,
    on=['PlayerId', 'ShotId'],
    how='inner'
)

print(f"Merged rows with combined exports: {len(test_merged)}")
print(f"Sample merged data:")
print(test_merged[['PlayerId', 'ShotId', 'feat_1']].head(10))


Merged rows with combined exports: 29
Sample merged data:
   PlayerId ShotId                                           feat_1
0  Player 1      3                         BallVeloRelease__zwithin
1  Player 1      4           timeToMaxHipExtensionDominantPostHitch
2  Player 1      5                         BallVeloRelease__zwithin
3  Player 1      8                         BallVeloRelease__zwithin
4  Player 1     10  timeToMaxShoulderExtensionDomDiffHitch__zwithin
5  Player 1     13  timeToMaxShoulderExtensionDomDiffHitch__zwithin
6  Player 1     14  timeToMaxShoulderExtensionDomDiffHitch__zwithin
7  Player 1     16                         BallVeloRelease__zwithin
8  Player 1     23                    HipAlignmentPreHitch__zwithin
9  Player 1     24                    HipAlignmentPreHitch__zwithin


In [13]:
# Save the combined exports to a clean CSV file
output_path = 'public/data/all_skeleton_exports.csv'
combined_exports_df.to_csv(output_path, index=False)
print(f"Saved {len(combined_exports_df)} rows to {output_path}")


Saved 10987 rows to public/data/all_skeleton_exports.csv


## 5. Analysis: Why Do Only Some Shots Have SHAP Corrections?

In [14]:
# Compare the total shots with shots that have SHAP corrections
print(f"Total shots in main CSV: {len(main_df)}")
print(f"Total unique players: {main_df['Name'].nunique()}")
print(f"Total SHAP corrections available: {len(combined_exports_df)}")
print(f"Percentage of shots with SHAP data: {len(combined_exports_df) / len(main_df) * 100:.1f}%")


Total shots in main CSV: 25868
Total unique players: 165
Total SHAP corrections available: 10987
Percentage of shots with SHAP data: 42.5%


In [15]:
# Analyze by made/missed status
print("\n--- Distribution by Shot Result ---")
made = main_df[main_df['Made'].astype(str).str.upper() == 'TRUE'].shape[0]
missed = main_df[main_df['Made'].astype(str).str.upper() == 'FALSE'].shape[0]
print(f"Made shots: {made}")
print(f"Missed shots: {missed}")

# See which players have SHAP data and how many corrections per player
print("\n--- SHAP Corrections by Player (top 10) ---")
shap_by_player = combined_exports_df['PlayerId'].value_counts()
print(shap_by_player.head(10))

print(f"\nPlayers with SHAP data: {len(shap_by_player)}")
print(f"Average corrections per player: {len(combined_exports_df) / len(shap_by_player):.1f}")



--- Distribution by Shot Result ---
Made shots: 16906
Missed shots: 8962

--- SHAP Corrections by Player (top 10) ---
PlayerId
Player 163    1436
Player 59      791
Player 102     697
Player 158     561
Player 98      397
Player 64      325
Player 116     253
Player 120     200
Player 125     141
Player 103     137
Name: count, dtype: int64

Players with SHAP data: 158
Average corrections per player: 69.5
